# Build the Performance Table
The Goal of this script is tack on performance metrics 

## Read in the data

In [ ]:
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.igd import IGD
from pymoo.indicators.gd import GD
import numpy as np
import pandas as pd

from matplotlib.animation import FuncAnimation
from IPython.display import HTML


year = 2000
run_file_name = r"/rdata/ian/pico/paperRuns/global_run_table.pkl"
global_pf_file_name = r"/rdata/ian/pico/paperRuns/global_pf.pkl"

full_run_tab = pd.read_pickle(run_file_name)
global_pf = pd.read_pickle(global_pf_file_name)

global_pf = global_pf.sort_values(by="yield")

gd_ind = GD(global_pf.loc[:,("irr_total", "yield")].values)



## Configuration Definition
What configurations do we want to measure against our baseline? 

In [ ]:
algorithm = ["pinsga2", "pinsga2", "pinsga2", "pinsga2", "nsga2"]
dm_range = ["25to30", "10to20", "20to30", "20to30", "None"]
pop_size = [30, 30, 100, 60, 120]

configurations = [
    np.all([
        full_run_tab['algorithm'] == algorithm[c], 
        full_run_tab['DM_range'] == dm_range[c],
        full_run_tab['pop_size'] == pop_size[c]
        ], axis=0) 
    for c in range(len(algorithm))  
]


### Performance metric functions
Different ways we can evaluate a given configuration 

#### Final Generation Generational Distance


In [ ]:
def func(df): 
    df['gd_final_gen'] = gd_ind(df.loc[:,('yield','irr_total')].values)
    return df.loc[:,('algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen')]


def finalGenGD(tab): 

    max_gen = max(tab['gen'])

    # Filter all but the final generations 
    tab = tab[tab["gen"] == max_gen]

    summary = tab.groupby(['algorithm', 'DM_range', 'pop_size', 'run']).apply(func)

    return summary.drop_duplicates(subset=['algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen'])





### Run the analysis 


In [ ]:

summarized  = None 

for config in configurations:

    current_summary = finalGenGD(full_run_tab[config])

    if summarized is None: 
        summarized = current_summary
    else: 
        summarized = pd.concat([summarized, current_summary])


summarized



